In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Create dataframe

data = {
        'Company':['GOOG','GOOG','MSFT','MSFT','FB','FB', 'FB'],
        'Person': ['Sam','Charlie','Amy','Vanessa','Sam','Sarah', 'John'],
        'Sales':  [200,120,340,124,243,350, 250],
        'Calls':  [60,190,2,100,700,423, 133]
        }

In [3]:
df = pd.DataFrame(data)
df

,Company,Person,Sales,Calls
0,GOOG,Sam,200,60
1,GOOG,Charlie,120,190
2,MSFT,Amy,340,2
3,MSFT,Vanessa,124,100
4,FB,Sam,243,700
5,FB,Sarah,350,423
6,FB,John,250,133


### What is the total Sales for all companies ?

In [4]:
df['Sales'].sum()

1627

### What is the total Sales per Company ?

In [5]:
df.groupby('Company')['Sales'].sum().sort_values(ascending= False)


Company
FB      843
MSFT    464
GOOG    320
Name: Sales, dtype: int64

### What is the Average number of calls per Company ?

In [6]:
df.groupby('Company')['Calls'].mean().sort_values(ascending= False).round(2)

Company
FB      418.67
GOOG    125.00
MSFT     51.00
Name: Calls, dtype: float64

### How Many employees per Company ?

In [7]:
df.groupby('Company')['Person'].count().sort_values(ascending= False)

Company
FB      3
GOOG    2
MSFT    2
Name: Person, dtype: int64

In [8]:
df['Company'].value_counts()

Company
FB      3
GOOG    2
MSFT    2
Name: count, dtype: int64

* Add missing value in Person column
    - How Many emplyees per Company ?

In [9]:
# Add a row with a missing value in the Person column

data = {
        'Company':['GOOG','GOOG','MSFT','MSFT','FB','FB', 'FB', 'GOOG'],
        'Person': ['Sam','Charlie','Amy','Vanessa','Sam','Sarah', 'John', np.nan],
        'Sales':  [200,120,340,124,243,350, 250, 300],
        'Calls':  [60,190,2,100,700,423, 133, 200]
        }

df_2 = pd.DataFrame(data)
df_2

,Company,Person,Sales,Calls
0,GOOG,Sam,200,60
1,GOOG,Charlie,120,190
2,MSFT,Amy,340,2
3,MSFT,Vanessa,124,100
4,FB,Sam,243,700
5,FB,Sarah,350,423
6,FB,John,250,133
7,GOOG,NaN,300,200


In [10]:
df_2.groupby('Company').size()

Company
FB      3
GOOG    3
MSFT    2
dtype: int64

In [11]:
df_2['Company'].value_counts()

Company
GOOG    3
FB      3
MSFT    2
Name: count, dtype: int64

In [12]:
# Side by side: GOOG shows 3 rows but only 2 known people

pd.DataFrame({
    'rows (size)': df_2.groupby('Company').size(),
    'known people (count)': df_2.groupby('Company')['Person'].count(),
    'unique people (nunique)': df_2.groupby('Company')['Person'].nunique(),
})

,rows (size),known people (count),unique people (nunique)
Company,,,
FB,3,3,3
GOOG,3,2,2
MSFT,2,2,2


## Pivoting

In [13]:
df_3 = pd.read_csv('dataset/gapminder-FiveYearData.csv')
df_3

,country,year,pop,continent,lifeExp,gdpPercap
0,Afghanistan,1952,8425333.0,Asia,28.801,779.445314
1,Afghanistan,1957,9240934.0,Asia,30.332,820.853030
2,Afghanistan,1962,10267083.0,Asia,31.997,853.100710
3,Afghanistan,1967,11537966.0,Asia,34.020,836.197138
4,Afghanistan,1972,13079460.0,Asia,36.088,739.981106
...,...,...,...,...,...,...
1699,Zimbabwe,1987,9216418.0,Africa,62.351,706.157306
1700,Zimbabwe,1992,10704340.0,Africa,60.377,693.420786
1701,Zimbabwe,1997,11404948.0,Africa,46.809,792.449960
1702,Zimbabwe,2002,11926563.0,Africa,39.989,672.038623


### What is the average GDP per Continent ?

In [14]:
df_3.groupby('continent')['gdpPercap'].mean().sort_values(ascending= False).round(2)

continent
Oceania     18621.61
Europe      14469.48
Asia         7902.15
Americas     7136.11
Africa       2193.75
Name: gdpPercap, dtype: float64

In [15]:
# population-weighted GDP per capita:
# total GDP (`gdpPercap * pop`) divided by total population.

df_3['gdp'] = df_3['gdpPercap'] * df_3['pop']

weighted = df_3.groupby('continent')[['gdp', 'pop']].sum()
weighted['gdpPercap_weighted'] = weighted['gdp'] / weighted['pop']

pd.DataFrame({
    'simple mean (per country)': df_3.groupby('continent')['gdpPercap'].mean(),
    'population-weighted': weighted['gdpPercap_weighted'],
}).sort_values('population-weighted', ascending=False).round(2)

,simple mean (per country),population-weighted
continent,,
Oceania,18621.61,21204.96
Europe,14469.48,15692.82
Americas,7136.11,15477.07
Asia,7902.15,2949.60
Africa,2193.75,2108.19


### What is the average GDP per Continent per Country?

In [16]:
df_3.groupby(['continent', 'country'])['gdpPercap'].mean().sort_values(ascending= False).round(2)

continent  country      
Asia       Kuwait           65332.91
Europe     Switzerland      27074.33
           Norway           26747.31
Americas   United States    26261.15
           Canada           22410.75
                              ...   
Africa     Mozambique         542.28
           Eritrea            541.00
           Ethiopia           509.12
           Burundi            471.66
Asia       Myanmar            439.33
Name: gdpPercap, Length: 142, dtype: float64

In [17]:
# global sort_values would mix the continents together
# top 5 per continent

by_country = (
    df_3.groupby(['continent', 'country'])['gdpPercap'].mean().round(2)
        .groupby(level='continent', group_keys=False)
        .apply(lambda s: s.sort_values(ascending=False))
)
by_country.groupby(level='continent').head(5)   

continent  country      
Africa     Libya            12013.58
           Gabon            11529.87
           South Africa      7247.43
           Botswana          5031.50
           Reunion           4898.40
Americas   United States    26261.15
           Canada           22410.75
           Puerto Rico      10863.16
           Venezuela        10088.52
           Argentina         8955.55
Asia       Kuwait           65332.91
           Saudi Arabia     20261.74
           Bahrain          18077.66
           Japan            17750.87
           Singapore        17425.38
Europe     Switzerland      27074.33
           Norway           26747.31
           Netherlands      21748.85
           Denmark          21671.82
           Germany          20556.68
Oceania    Australia        19980.60
           New Zealand      17262.62
Name: gdpPercap, dtype: float64

### What is the average GDP per Continent per year ?

In [18]:
df_3.groupby(['continent', 'year'])['gdpPercap'].mean().round(2)

continent  year
Africa     1952     1252.57
           1957     1385.24
           1962     1598.08
           1967     2050.36
           1972     2339.62
           1977     2585.94
           1982     2481.59
           1987     2282.67
           1992     2281.81
           1997     2378.76
           2002     2599.39
           2007     3089.03
Americas   1952     4079.06
           1957     4616.04
           1962     4901.54
           1967     5668.25
           1972     6491.33
           1977     7352.01
           1982     7506.74
           1987     7793.40
           1992     8044.93
           1997     8889.30
           2002     9287.68
           2007    11003.03
Asia       1952     5195.48
           1957     5787.73
           1962     5729.37
           1967     5971.17
           1972     8187.47
           1977     7791.31
           1982     7434.14
           1987     7608.23
           1992     8639.69
           1997     9834.09
           2002    10174.09
    

In [19]:
pd.pivot_table(df_3, index= 'continent', columns= 'year', values= 'gdpPercap').round(2)


year,1952,1957,1962,1967,1972,1977,1982,1987,1992,1997,2002,2007
continent,,,,,,,,,,,,
Africa,1252.57,1385.24,1598.08,2050.36,2339.62,2585.94,2481.59,2282.67,2281.81,2378.76,2599.39,3089.03
Americas,4079.06,4616.04,4901.54,5668.25,6491.33,7352.01,7506.74,7793.40,8044.93,8889.30,9287.68,11003.03
Asia,5195.48,5787.73,5729.37,5971.17,8187.47,7791.31,7434.14,7608.23,8639.69,9834.09,10174.09,12473.03
Europe,5661.06,6963.01,8365.49,10143.82,12479.58,14283.98,15617.90,17214.31,17061.57,19076.78,21711.73,25054.48
Oceania,10298.09,11598.52,12696.45,14495.02,16417.33,17283.96,18554.71,20448.04,20894.05,24024.18,26938.78,29810.19


In [20]:
df_3.groupby(['continent', 'year'])['gdpPercap'].mean().unstack().round(2)

year,1952,1957,1962,1967,1972,1977,1982,1987,1992,1997,2002,2007
continent,,,,,,,,,,,,
Africa,1252.57,1385.24,1598.08,2050.36,2339.62,2585.94,2481.59,2282.67,2281.81,2378.76,2599.39,3089.03
Americas,4079.06,4616.04,4901.54,5668.25,6491.33,7352.01,7506.74,7793.40,8044.93,8889.30,9287.68,11003.03
Asia,5195.48,5787.73,5729.37,5971.17,8187.47,7791.31,7434.14,7608.23,8639.69,9834.09,10174.09,12473.03
Europe,5661.06,6963.01,8365.49,10143.82,12479.58,14283.98,15617.90,17214.31,17061.57,19076.78,21711.73,25054.48
Oceania,10298.09,11598.52,12696.45,14495.02,16417.33,17283.96,18554.71,20448.04,20894.05,24024.18,26938.78,29810.19


In [21]:
pd.pivot_table(df_3, index=  'year', columns= 'continent', values= 'gdpPercap').round(2)

continent,Africa,Americas,Asia,Europe,Oceania
year,,,,,
1952,1252.57,4079.06,5195.48,5661.06,10298.09
1957,1385.24,4616.04,5787.73,6963.01,11598.52
1962,1598.08,4901.54,5729.37,8365.49,12696.45
1967,2050.36,5668.25,5971.17,10143.82,14495.02
1972,2339.62,6491.33,8187.47,12479.58,16417.33
1977,2585.94,7352.01,7791.31,14283.98,17283.96
1982,2481.59,7506.74,7434.14,15617.90,18554.71
1987,2282.67,7793.40,7608.23,17214.31,20448.04
1992,2281.81,8044.93,8639.69,17061.57,20894.05
